# 03 — Model Selection

Compare LightGBM, XGBoost, and CatBoost on the validation set.

In [9]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from training.preprocessing import build_preprocessor, engineer_features, ALL_FEATURES

TARGET = 'log_resale_price'

df = engineer_features(pd.read_csv('../../data/Resaleflatpricesbase.csv'))
train = df[df['transaction_year'] <= 2024]
val   = df[(df['transaction_year'] == 2025) & (df['transaction_month'] <= 9)]

preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(train[ALL_FEATURES], train[TARGET])
X_val   = preprocessor.transform(val[ALL_FEATURES])
y_train, y_val = train[TARGET], val[TARGET]

In [10]:
def evaluate(name, model, X, y_log):
    pred = np.expm1(model.predict(X))
    true = np.expm1(y_log)
    mae  = mean_absolute_error(true, pred)
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    mape = float(np.mean(np.abs((true - pred) / true)) * 100)
    r2   = r2_score(true, pred)
    print(f'{name:12s}: MAE={mae:,.0f} | RMSE={rmse:,.0f} | MAPE={mape:.2f}% | R²={r2:.4f}')
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

In [11]:
# LightGBM
lgbm = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1
)
lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)],
         callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
res_lgbm = evaluate('LightGBM', lgbm, X_val, y_val)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001244 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1321
[LightGBM] [Info] Number of data points in the train set: 196982, number of used features: 56
[LightGBM] [Info] Start training from score 13.080356
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0131295
[200]	valid_0's l2: 0.0105636
[300]	valid_0's l2: 0.00965066
[400]	valid_0's l2: 0.00914177
[500]	valid_0's l2: 0.00882293
[600]	valid_0's l2: 0.00864028
[700]	valid_0's l2: 0.00850731
[800]	valid_0's l2: 0.00839066
[900]	valid_0's l2: 0.00831325
[1000]	valid_0's l2: 0.00824993
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 0.00824993
LightGBM    : MAE=47,762 | RMSE=62,899 | MAPE=7.15% | R²=0.9030


In [12]:
# XGBoost
xgbm = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='rmse', early_stopping_rounds=50, random_state=42
)
xgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
res_xgb = evaluate('XGBoost', xgbm, X_val, y_val)

[0]	validation_0-rmse:0.38531
[100]	validation_0-rmse:0.11474
[200]	validation_0-rmse:0.10658
[300]	validation_0-rmse:0.10206
[400]	validation_0-rmse:0.09905
[500]	validation_0-rmse:0.09693
[600]	validation_0-rmse:0.09547
[700]	validation_0-rmse:0.09429
[800]	validation_0-rmse:0.09327
[900]	validation_0-rmse:0.09249
[999]	validation_0-rmse:0.09197
XGBoost     : MAE=48,299 | RMSE=63,245 | MAPE=7.24% | R²=0.9019


In [13]:
# Summary
import pandas as pd
results = pd.DataFrame([res_lgbm, res_xgb])
results.set_index('model').sort_values('MAE')

,MAE,RMSE,MAPE,R2
model,,,,
LightGBM,47762.314742,62899.159438,7.147586,0.902974
XGBoost,48298.865072,63244.917416,7.242067,0.901904
